In [ ]:
# EDA -- Dataset 1: PROMISE -- ShiftMetrics Analytics [CORREGIDO]
# Subcarpeta correcta: bug-data/ (metricas CK con nombres, columna bug)
# Fixes: sintaxis ck_cols, carga de todos los archivos, deteccion robusta


In [ ]:
# -- 0. Autenticacion GCP + Imports
from google.colab import auth
auth.authenticate_user()
print('OK Autenticado')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import storage
import io, os, re
from collections import Counter, defaultdict

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.3f}'.format)
sns.set_theme(style='darkgrid')

PROJECT = 'shiftmetrics-analytics'
BUCKET  = 'shiftmetrics-bronze'
print('OK Imports OK')


In [ ]:
# -- 1. Descubrir estructura de carpetas en Bronze/promise/
# CORRECCION: verificamos que bug-data/ exista antes de usarla
client = storage.Client(project=PROJECT)
bucket_obj = client.bucket(BUCKET)

all_blobs = list(bucket_obj.list_blobs(prefix='promise/'))
print('Total objetos bajo promise/:', len(all_blobs))

subfolders = defaultdict(list)
for b in all_blobs:
    parts = b.name.replace('promise/', '').split('/')
    sub = parts[0] if len(parts) > 1 else '(raiz)'
    subfolders[sub].append(b)

print('\nSubcarpetas encontradas:')
for sub, blbs in sorted(subfolders.items()):
    total_mb = sum(b.size for b in blbs) / (1024**2)
    csv_count = sum(1 for b in blbs if b.name.endswith('.csv'))
    print(f'  {sub:45s}  {len(blbs):4d} objetos  {total_mb:8.1f} MB  ({csv_count} CSVs)')


In [ ]:
# -- 2. Seleccionar subcarpeta bug-data (metricas CK nombradas)
# bug-data/  -> wmc, dit, noc, cbo, rfc, lcom, ca, ce, npm, lcom3, loc,
#               dam, moa, mfa, cam, ic, cbm, amc, bug  <- columna target
# AST_encoding/ -> columnas numericas anonimas (NO utiles para ML)
PREFIX = 'promise/PROMISE-backup/bug-data/'

blobs = list(bucket_obj.list_blobs(prefix=PREFIX))
csv_blobs = [b for b in blobs if b.name.endswith('.csv')]

print('Subcarpeta:', PREFIX)
print('Total archivos CSV:', len(csv_blobs))
print('\nPrimeros 20:')
for b in csv_blobs[:20]:
    print(f'  {os.path.basename(b.name):40s}  {b.size/1024:.1f} KB')


In [ ]:
# -- 3. Cargar TODOS los archivos CSV
# CORRECCION: antes solo se cargaban 10. Ahora cargamos todos.
dfs = []
errors = []

for blob in csv_blobs:
    try:
        data = blob.download_as_bytes()
        for enc in ['utf-8', 'latin-1', 'cp1252']:
            try:
                df = pd.read_csv(io.BytesIO(data), encoding=enc, low_memory=False)
                df['_source_file'] = os.path.basename(blob.name)
                dfs.append(df)
                print(f'OK {os.path.basename(blob.name):45s} {df.shape[0]:6d} filas x {df.shape[1]:3d} cols')
                break
            except UnicodeDecodeError:
                continue
    except Exception as e:
        errors.append((blob.name, str(e)))
        print(f'ERR {blob.name}: {e}')

print(f'\nCargados: {len(dfs)} archivos | Errores: {len(errors)}')


In [ ]:
# -- 4. Esquemas y columnas comunes
print('=== ESQUEMAS PRIMEROS 5 ARCHIVOS ===\n')
for df in dfs[:5]:
    fname = df['_source_file'].iloc[0]
    cols = [c for c in df.columns if not c.startswith('_')]
    print(f'Archivo: {fname}')
    print(f'   Filas: {len(df):,} | Columnas: {len(cols)}')
    print(f'   Cols: {cols}')
    print()

all_col_sets = [set(df.columns) - {'_source_file'} for df in dfs]
common_cols = set.intersection(*all_col_sets)
print(f'Columnas comunes a los {len(dfs)} archivos: {len(common_cols)}')
print(sorted(common_cols))


In [ ]:
# -- 5. Detectar columna target
TARGET_CANDIDATES = ['bug', 'defects', 'defect', 'class', 'label',
                     'Defective', 'defective', 'bugs', 'fault', 'faults', 'Fault']

print('=== COLUMNAS TARGET POR ARCHIVO (primeros 5) ===\n')
for df in dfs[:5]:
    fname = df['_source_file'].iloc[0]
    cols_lower = {c.lower(): c for c in df.columns}
    found = []
    for tc in TARGET_CANDIDATES:
        if tc.lower() in cols_lower:
            col = cols_lower[tc.lower()]
            found.append((col, df[col].value_counts().to_dict()))
    print(f'Archivo: {fname}')
    if found:
        for col, vals in found:
            print(f'   -> {col}: {vals}')
    else:
        print('   SIN columna target reconocida')
    print()


In [ ]:
# -- 6. DataFrame unificado
df_all = pd.concat([df[list(common_cols) + ['_source_file']] for df in dfs], ignore_index=True)
print(f'DataFrame unificado: {df_all.shape[0]:,} filas x {df_all.shape[1]} cols')
print(f'Proyectos unicos: {df_all["_source_file"].nunique()}')
print(df_all['_source_file'].value_counts().to_string())


In [ ]:
# -- 7. Balance de clases
TARGET_CANDIDATES = ['bug', 'defects', 'defect', 'class', 'label', 'Defective', 'defective', 'bugs', 'fault', 'faults']
target_col = None
for tc in TARGET_CANDIDATES:
    if tc in df_all.columns:
        target_col = tc
        break

if target_col:
    print(f'OK Columna target: {target_col}')
    target_binary = pd.to_numeric(df_all[target_col], errors='coerce')
    df_all['_is_defective'] = (target_binary > 0).astype(int)
    vc = df_all['_is_defective'].value_counts()
    total = len(df_all)
    print(f'Balance global:')
    print(f'  Limpio     (0): {vc.get(0,0):,} ({vc.get(0,0)/total*100:.1f}%)')
    print(f'  Defectuoso (1): {vc.get(1,0):,} ({vc.get(1,0)/total*100:.1f}%)')
    fig, ax = plt.subplots(figsize=(6, 4))
    vc.plot(kind='bar', ax=ax, color=['#2ecc71','#e74c3c'], edgecolor='black')
    ax.set_title('Balance de clases -- PROMISE bug-data')
    ax.set_xticklabels(['Limpio', 'Defectuoso'], rotation=0)
    for p in ax.patches:
        ax.annotate(f'{int(p.get_height()):,}\n({p.get_height()/total*100:.1f}%)',
                    (p.get_x()+p.get_width()/2., p.get_height()),
                    ha='center', va='bottom', fontsize=10)
    plt.tight_layout(); plt.savefig('promise_class_balance.png', dpi=120); plt.show()
else:
    print('WARN No se detecto columna target. Cols:', list(df_all.columns))


In [ ]:
# -- 8. Metricas CK (Chidamber & Kemerer) -- CORREGIDO
# CORRECCION PRINCIPAL:
# ANTES (con error): hal_cols = [c for c in df_all.columns if c.lower() in [m.lower() for m in]]
# DESPUES (correcto): completar 'for m in CK_METRICS' + fix de indentacion

CK_METRICS = ['wmc', 'dit', 'noc', 'cbo', 'rfc', 'lcom', 'ca', 'ce',
              'npm', 'lcom3', 'loc', 'dam', 'moa', 'mfa', 'cam',
              'ic', 'cbm', 'amc', 'max_cc', 'avg_cc']

ck_cols = [c for c in df_all.columns if c.lower() in [m.lower() for m in CK_METRICS]]
print(f'Columnas CK encontradas ({len(ck_cols)}): {ck_cols}')

if ck_cols:
    print('\nEstadisticas CK:')
    print(df_all[ck_cols].describe().T[['count','mean','std','min','50%','max']])
    ncols = min(len(ck_cols), 6)
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()
    for i, col in enumerate(ck_cols[:6]):
        data = np.log1p(df_all[col].dropna().clip(lower=0))
        axes[i].hist(data, bins=50, color='#3498db', edgecolor='black', alpha=0.7)
        axes[i].set_title(f'{col} (log1p)')
    for j in range(ncols, 6):
        axes[j].set_visible(False)
    plt.suptitle('Distribucion Metricas CK -- PROMISE bug-data', fontsize=14)
    plt.tight_layout(); plt.savefig('promise_ck_dist.png', dpi=120); plt.show()
else:
    print('WARN No se encontraron cols CK. Cols del DataFrame:', list(df_all.columns))


In [ ]:
# -- 9. Metricas McCabe
MCCABE_COLS = ['v(g)', 'ev(g)', 'iv(g)', 'branchCount', 'max_cc', 'avg_cc']
mcc_cols = [c for c in df_all.columns
            if c.lower() in [x.lower() for x in MCCABE_COLS]
            or 'cyclo' in c.lower() or 'mccabe' in c.lower()]
print(f'Columnas McCabe ({len(mcc_cols)}): {mcc_cols}')
if mcc_cols:
    print(df_all[mcc_cols].describe().T[['count','mean','std','min','50%','max']])
    fig, axes = plt.subplots(1, min(len(mcc_cols), 3), figsize=(15, 4))
    if len(mcc_cols) == 1: axes = [axes]
    for i, col in enumerate(mcc_cols[:3]):
        data = np.log1p(df_all[col].dropna().clip(lower=0))
        axes[i].hist(data, bins=50, color='#e67e22', alpha=0.7)
        axes[i].set_title(f'{col} (log1p)')
    plt.tight_layout(); plt.savefig('promise_mccabe_dist.png', dpi=120); plt.show()
else:
    print('WARN No se encontraron cols McCabe.')


In [ ]:
# -- 10. Correlacion metricas CK vs defectos
if target_col and ck_cols:
    target_binary = df_all['_is_defective']
    corrs = {}
    for col in ck_cols:
        try:
            r = pd.to_numeric(df_all[col], errors='coerce').corr(target_binary)
            if not np.isnan(r): corrs[col] = r
        except: pass
    corr_series = pd.Series(corrs).sort_values(ascending=False)
    print('Correlacion Pearson -- metricas CK vs defectuoso:')
    print(corr_series.to_string())
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ['#e74c3c' if v > 0 else '#2ecc71' for v in corr_series.values]
    corr_series.plot(kind='barh', ax=ax, color=colors)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_title('Correlacion metricas CK vs defectos -- PROMISE')
    ax.set_xlabel('Pearson r')
    plt.tight_layout(); plt.savefig('promise_correlations.png', dpi=120); plt.show()
else:
    print('WARN Requiere target_col y ck_cols')


In [ ]:
# -- 11. Valores nulos
null_pct = (df_all.isnull().sum() / len(df_all) * 100).sort_values(ascending=False)
null_pct = null_pct[null_pct > 0]
print(f'Columnas con nulos: {len(null_pct)} / {len(df_all.columns)}')
if len(null_pct) > 0:
    print(null_pct.to_string())
    fig, ax = plt.subplots(figsize=(10, max(4, len(null_pct)*0.4)))
    null_pct.plot(kind='barh', ax=ax, color='#e74c3c')
    ax.set_title('% valores nulos -- PROMISE')
    plt.tight_layout(); plt.savefig('promise_nulls.png', dpi=120); plt.show()
else:
    print('OK Sin valores nulos detectados')


In [ ]:
# -- 12. Defect Density por proyecto
if '_is_defective' in df_all.columns:
    density_by_file = (df_all.groupby('_source_file')
                       .agg(total_modules=('_is_defective','count'),
                            defective_modules=('_is_defective','sum'))
                       .assign(defect_density=lambda x: x['defective_modules']/x['total_modules'])
                       .sort_values('defect_density', ascending=False))
    print('Defect Density por archivo:')
    print(density_by_file.to_string())
    dd = density_by_file['defect_density']
    print(f'\np25={dd.quantile(.25):.3f}  median={dd.median():.3f}  p75={dd.quantile(.75):.3f}  max={dd.max():.3f}')
    fig, ax = plt.subplots(figsize=(14, 8))
    density_by_file['defect_density'].sort_values().plot(kind='barh', ax=ax, color='#9b59b6')
    ax.set_title('Defect Density por version -- PROMISE')
    plt.tight_layout(); plt.savefig('promise_defect_density.png', dpi=120); plt.show()
else:
    print('WARN Ejecutar celda 7 primero')


In [ ]:
# -- 13. Resumen EDA
print('=' * 65)
print('RESUMEN EDA -- PROMISE bug-data (CORREGIDO)')
print('=' * 65)
print('Total CSV en bug-data/:', len(csv_blobs))
print('Archivos cargados:', len(dfs))
print('Filas totales:', len(df_all))
print('Columnas comunes:', len(common_cols))

if target_col:
    vc = df_all['_is_defective'].value_counts()
    total = len(df_all)
    print('Target:', target_col, '-> binarizado en _is_defective')
    print('  Limpio (0):', vc.get(0,0), '/', total)
    print('  Defectuoso (1):', vc.get(1,0), '/', total)

if 'density_by_file' in dir():
    dd = density_by_file['defect_density']
    print('Defect Density: p25=%.3f median=%.3f p75=%.3f max=%.3f' % (dd.quantile(.25), dd.median(), dd.quantile(.75), dd.max()))

print('Cols CK:', ck_cols if 'ck_cols' in dir() else 'N/A')
print('Cols McCabe:', mcc_cols if 'mcc_cols' in dir() else 'N/A')

print('\n--- PARAMETROS PARA JOB SILVER PROMISE ---')
print('Subcarpeta: promise/PROMISE-backup/bug-data/')
print('Columnas a conservar: wmc, dit, noc, cbo, rfc, lcom, ca, ce, npm, lcom3, loc, dam, moa, mfa, cam, ic, cbm, amc, bug')
print('Target: binarizar bug -> (bug > 0) = 1')
print('Agregar: proyecto y version desde nombre de archivo')
print('Nulos: imputar con mediana por proyecto')
print('Particion Parquet: por proyecto')

print('\n--- PARAMETROS PARA DATASET 5 (SIMULADOR) ---')
print('tasa_defectos_por_modulo: usar median de Defect Density')
print('Rangos CK: ver describe() de celda 8')
print('Proyectos referencia: ant, camel, jedit, log4j, lucene, poi')

print('\nOK EDA PROMISE completado y corregido')
